# 2 段構成の評価: VideoJev の候補区間 → 学習済み Qwen3-VL で当て直し（Colab A100）

1 段目は VideoJev-4B の予測（Drive の `results/videojev-4b-p`、無ければ `videojev-4b`）。2 段目は候補区間の前後 30 秒（最短 60 秒）を切り抜き、学習済み Qwen3-VL-4B（`grpo3_time/step_00400`）が 32 フレームで見て時刻を当て直す。
採点は同じ 722 問・同じ許容幅。VideoJev 単体 83.4%、Qwen 単体 69.9% と並べる。詳細は `docs/videojev_eval.md`。

前提: Drive の `MyDrive/movie_llm/` に `videos_2fps_s360.zip` と `grpo3_time/step_00400/` がある。約 1 時間。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. リポジトリと依存パッケージ

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "transformers>=5.0" "accelerate>=1.0" "peft>=0.15" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "python-dotenv" "openai"
!pip uninstall -y -q torchao
!pip install -q -e . --no-deps

## 2. Drive、動画、1 段目の予測

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
ADAPTER = f'{DRIVE}/grpo3_time/step_00400'
import os
!mkdir -p /content/videos_small
!unzip -q -o {DRIVE}/videos_2fps_s360.zip -d /content/videos_small
VJ = f'{DRIVE}/results/videojev-4b-p/predictions.jsonl'
if not os.path.exists(VJ):
    VJ = f'{DRIVE}/results/videojev-4b/predictions.jsonl'
print('動画', len(os.listdir('/content/videos_small')), '本', '| LoRA', os.path.exists(ADAPTER), '| 1 段目', VJ)

## 3. 動作確認（5 問）

`vj=候補区間 win=切り抜き → 当て直した時刻 gold=正解区間` が出ていれば OK。

In [ ]:
%cd /content/movie_llm
!rm -rf /content/ts_smoke
!python scripts/jvqa_two_stage.py --vj-predictions {VJ} --name two_stage-4b --model Qwen/Qwen3-VL-4B-Instruct --video-dir /content/videos_small --adapter {ADAPTER} --out-dir /content/ts_smoke --limit 5

## 4. 全 722 問（約 1 時間。切れたら同じセルを再実行で続きから）

In [ ]:
%cd /content/movie_llm
!python scripts/jvqa_two_stage.py --vj-predictions {VJ} --name two_stage-4b --model Qwen/Qwen3-VL-4B-Instruct --video-dir /content/videos_small --adapter {ADAPTER} --out-dir {DRIVE}/results
!python scripts/jvqa_eval_time.py --predictions {DRIVE}/results/two_stage-4b/predictions.jsonl --name two_stage-4b --out-root {DRIVE}/results

終わったら「終わった」と言う。結果は Drive の `results/two_stage-4b/` から取る。